In [1]:
import pandas as pd
# import data
import kagglehub

# path = kagglehub.dataset_download("stealthtechnologies/employee-attrition-dataset")
# train_set = pd.read_csv(path+"/train.csv")
# test_set = pd.read_csv(path+"/test.csv")
train_set = pd.read_csv('data/train.csv')
test_set = pd.read_csv('data/test.csv')
dataset = pd.concat([train_set, test_set], axis=0, ignore_index=True)
dataset.head()

,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,...,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,8410,31,Male,19,Education,5390,Excellent,Medium,Average,2,...,0,Mid,Medium,89,No,No,No,Excellent,Medium,Stayed
1,64756,59,Female,4,Media,5534,Poor,High,Low,3,...,3,Mid,Medium,21,No,No,No,Fair,Low,Stayed
2,30257,24,Female,10,Healthcare,8159,Good,High,Low,0,...,3,Mid,Medium,74,No,No,No,Poor,Low,Stayed
3,65791,36,Female,7,Education,3989,Good,High,High,1,...,2,Mid,Small,50,Yes,No,No,Good,Medium,Stayed
4,65026,56,Male,41,Education,4821,Fair,Very High,Average,0,...,0,Senior,Medium,68,No,No,No,Fair,Medium,Stayed


In [2]:
ordinal_features = set(['Gender', 'Attrition', 'Work-Life Balance', 'Job Satisfaction', 'Performance Rating', 'Education Level', 'Job Level', 'Company Size', 'Company Reputation', 'Employee Recognition',])
unordinal_features = set(dataset.select_dtypes(include=['object']).columns) - ordinal_features

# print unique values of ordinal features
# for feature in ordinal_features:
#     print(f"{feature}: {dataset[feature].unique()}")

In [3]:
# ordinal features
dict_ordinal = {
    'Gender': {'Male': 0, 'Female': 1},
    'Employee Recognition': {'Low': 0, 'Medium': 1, 'High': 2, 'Very High': 3},
    'Company Size': {'Small': 0, 'Medium': 1, 'Large': 2},
    'Job Satisfaction': {'Low': 0, 'Medium': 1, 'High': 2, 'Very High': 3},
    'Performance Rating': {'Low': 0, 'Below Average': 1, 'Average': 2, 'High': 3},
    'Company Reputation': {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3},
    'Education Level': {'High School': 0, 'Associate Degree': 1, 'Bachelor’s Degree': 2, 'Master’s Degree': 3, 'PhD': 4},
    'Job Level': {'Entry': 0, 'Mid': 1, 'Senior': 2},
    'Work-Life Balance': {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3},
    'Attrition': {'Left': 0, 'Stayed': 1}
}
for feature in ordinal_features:
    dataset[feature] = dataset[feature].map(dict_ordinal[feature])

# unordinal features
for column in unordinal_features:
    dataset = dataset.join(pd.get_dummies(dataset[column], prefix=column))
    dataset.drop(labels=column, axis=1, inplace=True)

In [4]:
# use the first 80% of the dataset as training set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(dataset.drop(labels='Attrition', axis=1), dataset['Attrition'], test_size=0.2, shuffle=True)

# use knn, logistic regression, MLP
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score

import math
knn = KNeighborsClassifier(n_neighbors=math.ceil(math.sqrt(len(X_train))))
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print(f"KNN accuracy: {accuracy_score(y_test, y_pred)}")
print(f"KNN precision: {precision_score(y_test, y_pred)}")

logistic = LogisticRegression(max_iter=1000)
logistic.fit(X_train, y_train)
y_pred = logistic.predict(X_test)
print(f"Logistic regression accuracy: {accuracy_score(y_test, y_pred)}")
print(f"Logistic regression precision: {precision_score(y_test, y_pred)}")

mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)
print(f"MLP accuracy: {accuracy_score(y_test, y_pred)}")
print(f"MLP precision: {precision_score(y_test, y_pred)}")

# as expected, they all have the worse performance, and lr can't even converge
# strange bias to the positive class in MLP though

KNN accuracy: 0.5141610738255034
KNN precision: 0.5201240986080832


/home/b/myenv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic regression accuracy: 0.7364429530201342
Logistic regression precision: 0.736585974544547
MLP accuracy: 0.5902684563758389
MLP precision: 0.5596716947648624


In [5]:
# check with scaled data
from sklearn.preprocessing  import StandardScaler, MinMaxScaler
sc = StandardScaler()
mc = MinMaxScaler()
# scale ordinal features
dataset[list(ordinal_features - {'Gender', 'Attrition'})] = mc.fit_transform(dataset[list(ordinal_features - {'Gender', 'Attrition'})])
# scale continous features
dataset[['Years at Company', 'Monthly Income', 'Distance from Home', 'Age', 'Number of Promotions', 'Number of Dependents']] = sc.fit_transform(dataset[['Years at Company', 'Monthly Income', 'Distance from Home', 'Age', 'Number of Promotions', 'Number of Dependents']])

X_train, X_test, y_train, y_test = train_test_split(dataset.drop(labels='Attrition', axis=1), dataset['Attrition'], test_size=0.2, shuffle=True)

knn = KNeighborsClassifier(n_neighbors=math.ceil(math.sqrt(len(X_train))))
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print(f"KNN accuracy: {accuracy_score(y_test, y_pred)}")
print(f"KNN precision: {precision_score(y_test, y_pred)}")

logistic = LogisticRegression(max_iter=1000)
logistic.fit(X_train, y_train)
y_pred = logistic.predict(X_test)
print(f"Logistic regression accuracy: {accuracy_score(y_test, y_pred)}")
print(f"Logistic regression precision: {precision_score(y_test, y_pred)}")

mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)
print(f"MLP accuracy: {accuracy_score(y_test, y_pred)}")
print(f"MLP precision: {precision_score(y_test, y_pred)}")
                      

KNN accuracy: 0.5148322147651007
KNN precision: 0.5255093305940763


/home/b/myenv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic regression accuracy: 0.7373154362416108
Logistic regression precision: 0.747625680638217
MLP accuracy: 0.5189261744966442
MLP precision: 0.8376156217882836
